# Spam Classification

## Spam Classification (Intro)

P(spam|Features) = (P())

## The Spam Dataset

In [1]:
# Download the dataset and check the extracted files

import requests
import zipfile
import io
import os

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

response = requests.get(url)
if response.status_code == 200:
    print("Download successfull")
else:
    print("Failed to download the dataset")

directory_name = "sms_spam_collection"

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall(directory_name)
    print("Extraction successfull")

extracted = os.listdir(directory_name)
print("Extracted files:", extracted)

Download successfull
Extraction successfull
Extracted files: ['readme', 'SMSSpamCollection']


In [2]:
# load dataset into pandas

import pandas as pd

data_path = directory_name+"/SMSSpamCollection"

df = pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label", "message"],
)
   

In [3]:
# display basic information about the dataset

print("--------------- HEAD ---------------")
print(df.head())
print("--------------- DESCRIBE ---------------")
print(df.describe())
print("--------------- INFO ---------------")
print(df.info())

--------------- HEAD ---------------
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
--------------- DESCRIBE ---------------
       label                 message
count   5572                    5572
unique     2                    5169
top      ham  Sorry, I'll call later
freq    4825                      30
--------------- INFO ---------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


In [4]:
# clean the data

# check for missing values
print("Missing values:\n", df.isnull().sum())

# check duplicates
print("Duplicate entries:", df.duplicated().sum())

# remove duplicates if any
df = df.drop_duplicates()


Missing values:
 label      0
message    0
dtype: int64
Duplicate entries: 403


## Preprocesssing the Spam Dataset

In [ ]:
# download NLTK data files
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

print("=== Before any Preprocessing ===")
print(df.head(5))


In [6]:
# convert all text to lowercase (reduces dimensionality, as the model treats words equally regardles of the original casing)

df["message"] = df["message"].str.lower()
print("\n=== After Lowercasing ===")
print(df["message"].head(5))





=== After Lowercasing ===
0    go until jurong point, crazy.. available only ...
1                        ok lar... joking wif u oni...
2    free entry in 2 a wkly comp to win fa cup fina...
3    u dun say so early hor... u c already then say...
4    nah i don't think he goes to usf, he lives aro...
Name: message, dtype: object


In [7]:
# removing non-essential punctuation and numbers -- keep usefull symbols such as $ and !
import re

df["message"] = df["message"].apply(lambda x: re.sub(r"[^a-z\s$!]", "", x))
print("\n=== After removing punctuation & numbers (except $ and !) ===")
print(df["message"].head(5))


=== After removing punctuation & numbers (except $ and !) ===
0    go until jurong point crazy available only in ...
1                              ok lar joking wif u oni
2    free entry in  a wkly comp to win fa cup final...
3          u dun say so early hor u c already then say
4    nah i dont think he goes to usf he lives aroun...
Name: message, dtype: object


In [8]:
# split each message into individual tokens -- preparing for further analysis and removing stop words and applying stemming

from nltk.tokenize import word_tokenize

df["message"] = df["message"].apply(word_tokenize)
print("\n=== After Tokenization ===")
print(df["message"].head(5))


=== After Tokenization ===
0    [go, until, jurong, point, crazy, available, o...
1                       [ok, lar, joking, wif, u, oni]
2    [free, entry, in, a, wkly, comp, to, win, fa, ...
3    [u, dun, say, so, early, hor, u, c, already, t...
4    [nah, i, dont, think, he, goes, to, usf, he, l...
Name: message, dtype: object


In [9]:
# define a set of english stop words and remove them from the tokens
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))
df["message"] = df["message"].apply(lambda x: [word for word in x if word not in stop_words])
print("\n=== After removing stop words ===")
print(df["message"].head(5))



=== After removing stop words ===
0    [go, jurong, point, crazy, available, bugis, n...
1                       [ok, lar, joking, wif, u, oni]
2    [free, entry, wkly, comp, win, fa, cup, final,...
3        [u, dun, say, early, hor, u, c, already, say]
4    [nah, dont, think, goes, usf, lives, around, t...
Name: message, dtype: object


In [10]:
# stem each token to reduce words to their base form
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
df["message"] = df["message"].apply(lambda x: [stemmer.stem(word) for word in x])
print("\n=== After Stemming ===")
print(df["message"].head(5))


=== After Stemming ===
0    [go, jurong, point, crazi, avail, bugi, n, gre...
1                         [ok, lar, joke, wif, u, oni]
2    [free, entri, wkli, comp, win, fa, cup, final,...
3        [u, dun, say, earli, hor, u, c, alreadi, say]
4    [nah, dont, think, goe, usf, live, around, tho...
Name: message, dtype: object


In [11]:
# rejoin tokens into a single string, joined by spaces, for feature extraction
# many ML algorithms and vectorization techniques (e.g, TF-IDF) work best with raw text strings

df["message"] = df["message"].apply(lambda x: " ".join(x))
print("\n=== After Joining tokens back into strings ===")
print(df["message"].head(5))


=== After Joining tokens back into strings ===
0    go jurong point crazi avail bugi n great world...
1                                ok lar joke wif u oni
2    free entri wkli comp win fa cup final tkt st m...
3                  u dun say earli hor u c alreadi say
4            nah dont think goe usf live around though
Name: message, dtype: object


## Feature Extraction

In [12]:
# using CountVectorizer for the Bag-of-Words Approach


from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    min_df=1, # a term must appear in ≥ 1 document to be included (filters terms that are too rare)
    max_df=0.9, # exclude terms that appear in ≥ 90% od the documents (filters terms that are too common)
    ngram_range=(1, 2) # capture both unigrams and bigrams
)

# fit and transform the message column
X = vectorizer.fit_transform(df["message"])

# labels (targe variable) -- convert to 0 and 1
y = df["label"].apply(lambda x: 1 if x == "spam" else 0)


## Training and Evaluation (Spam Detection)

In [13]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Use pipeline to ensure that vectorization is consistently applied throughout all the hyperparameter tunnings/combinations
pipeline = Pipeline(
    [
        ("vectorizer", vectorizer),
        ("classifier", MultinomialNB())
    ]
)



In [14]:
# Grid search for hyperparameter tuning
param_grid = {
    "classifier__alpha": [0.01, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 1.0]
}

# grid search with 5-fold cross-validation, and use f1-score as reference performance metric
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5, 
    scoring="f1"
)

grid_search.fit(df["message"], y)

best_model = grid_search.best_estimator_
print("Best model parameters:", grid_search.best_params_)
print("Best model score:", grid_search.best_score_)

Best model parameters: {'classifier__alpha': 0.25}
Best model score: 0.9284151296453353


In [15]:
# Example SMS messages for evaluation
new_messages = [
    "Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/1234 to claim now.",
    "Hey, are we still meeting up for lunch today?",
    "Urgent! Your account has been compromised. Verify your details here: www.fakebank.com/verify",
    "Reminder: Your appointment is scheduled for tomorrow at 10am.",
    "FREE entry in a weekly competition to win an iPad. Just text WIN to 80085 now!",
]

In [16]:
import numpy as np
import re

def preprocess_message(message):
    message = message.lower()
    message = re.sub(r"[^a-z\s$!]", "", message)
    tokens = word_tokenize(message)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)

In [17]:
# preprocess and vectorize messages
processed_messages = [preprocess_message(msg) for msg in new_messages]

# transform preprocessed messages into feature vectors
X_new = best_model.named_steps["vectorizer"].transform(processed_messages)


In [18]:
# predict with the (best) trained classifier
predictions = best_model.named_steps["classifier"].predict(X_new)
predictions_probs = best_model.named_steps["classifier"].predict_proba(X_new)

In [19]:
# Display the predictions and probabilities for each message evaluated

for i, msg in enumerate(new_messages):
    prediciton = "Spam" if predictions[i] == 1 else "Not-Spam"
    spam_probability = predictions_probs[i][1] # probability of being spam
    ham_probability = predictions_probs[i][0] # probability of not being spam

    print(f"Message: {msg}")
    print(f"Prediction: {prediciton}")
    print(f"Spam Probability: {spam_probability:.2f}")
    print(f"Not-Spam Probability: {ham_probability:.2f}")
    print("-" * 50)

Message: Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/1234 to claim now.
Prediction: Spam
Spam Probability: 1.00
Not-Spam Probability: 0.00
--------------------------------------------------
Message: Hey, are we still meeting up for lunch today?
Prediction: Not-Spam
Spam Probability: 0.00
Not-Spam Probability: 1.00
--------------------------------------------------
Message: Urgent! Your account has been compromised. Verify your details here: www.fakebank.com/verify
Prediction: Spam
Spam Probability: 0.96
Not-Spam Probability: 0.04
--------------------------------------------------
Message: Reminder: Your appointment is scheduled for tomorrow at 10am.
Prediction: Not-Spam
Spam Probability: 0.00
Not-Spam Probability: 1.00
--------------------------------------------------
Message: FREE entry in a weekly competition to win an iPad. Just text WIN to 80085 now!
Prediction: Spam
Spam Probability: 1.00
Not-Spam Probability: 0.00
----------------------------------

In [20]:
# save the trained model
import joblib
model_filename = "spam_detection_model.joblib"
joblib.dump(best_model, model_filename)
print(f"Model saved to {model_filename}")

Model saved to spam_detection_model.joblib


## Model Evaluation

In [ ]:
# upload to HTB pay